# Cooperation for compounds from other members: consumption overlap

In [1]:
import pandas as pd
import json
import seaborn as sns
import numpy as np

import networkx as nx

from pyvis.network import Network

In [2]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")

In [3]:
with open("../data/Venturelli2018/generated/metid2metname.json") as json_file:
    metid2metname = json.load(json_file)

In [4]:
communities_12_exp_abundance = pd.read_csv("../data/Venturelli2018/generated/communities_12_exp_abundance.tsv",sep="\t",index_col=[0,1])

In [5]:
m9_mets = list(media_db[media_db.medium=="M9[-O2]"].compound.values)
m9_mets.remove("so4")
m9_mets.remove("nh4")
m9_mets.remove("glc__D")

m9_mets = ["M_"+met+"__91__e__93__" for met in m9_mets]

In [6]:
results_part1 = pd.read_csv("../output/Venturelli2018/results_communities_12_1000s_part1.tsv",sep="\t")
results_part2 = pd.read_csv("../output/Venturelli2018/results_communities_12_1000s_part2.tsv",sep="\t")
results = pd.concat([results_part1,results_part2]).reset_index(drop=True)

results["mass_rate*frequency"] = results["frequency"]*results["mass_rate"]

In [7]:
results = results[~results.compound.isin(m9_mets)].copy()

In [8]:
def centrality_metrics(network_data,cutoff=0,SP_centrality=True):
    """
    SP_centrality: centrality meassures based on shortest path
    """
    # Betweenness and closeness
    G = nx.Graph()
    G.add_nodes_from(list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique()))))
    
    if SP_centrality:
        edges = [(index[0],index[1],{"weight":1/data} ) for index,data in network_data.items() if data>cutoff]
    
    else:
        edges = [(index[0],index[1],{"weight":data} ) for index,data in network_data.items() if data>cutoff]
        
    G.add_edges_from(edges)
    
    if SP_centrality:
        return pd.DataFrame({"betweeness (1/data)":nx.centrality.betweenness_centrality(G,weight="weight"),
                     "closeness (1/data)":nx.centrality.closeness_centrality(G,distance="weight")})
    else:
        return pd.DataFrame({"degree (data)":dict(G.degree(weight="weight")),
                 "eigenvector (data)":nx.centrality.eigenvector_centrality(G,weight="weight")})


In [9]:
## From https://stackoverflow.com/questions/25668828/how-to-create-colour-gradient-in-python
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

def colorFader(c1,c2,mix=0): #fade (linear interpolate) from color c1 (at mix=0) to c2 (mix=1)
    c1=np.array(mpl.colors.to_rgb(c1))
    c2=np.array(mpl.colors.to_rgb(c2))
    return mpl.colors.to_hex((1-mix)*c1 + mix*c2)


In [10]:
b1='#f8cdcf' #bright pink
b2='#fa4f56' #red

c1='#ffffff' #white
c2='#0b5394' #blue

In [82]:
def frequency_network(network_data,closeness_centrality_met,cutoff=0):
    net = Network(directed=True)

    nodes = list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique())))
    color = [colorFader(b1,b2,closeness_centrality_met[node]) for node in nodes]
    value = [(closeness_centrality_met[node]*10)**2 for node in nodes]
    net.add_nodes(nodes,color=color,value=value)
    
    
    maximum_corr = network_data.max()
    for index,value in network_data.items():
        if value>cutoff:
            net.add_edge(index[0],index[1],width=(value**2)*10,color=colorFader(c1,c2,value/maximum_corr)) 

    net.barnes_hut(gravity=-1000, central_gravity=0.1, spring_length=150, spring_strength=0.0015, damping=0.09, overlap=0.8)


    template = net.templateEnv.get_template(net.path)
    net.template = template
    return net

### Average frequency

In [103]:
def frequency_network(network_data,closeness_centrality_met,cutoff=0):
    net = Network(directed=True)

    nodes = list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique())))
    color = [colorFader(b1,b2,closeness_centrality_met[node]) for node in nodes]
    value = [(closeness_centrality_met[node]*10)**2 for node in nodes]
    net.add_nodes(nodes,color=color,value=value)
    
    
    maximum_corr = network_data.max()
    for index,value in network_data.items():
        if value>cutoff:
            net.add_edge(index[0],index[1],width=(value**2)*100,color=colorFader(c1,c2,value/maximum_corr)) 

    net.barnes_hut(gravity=-1000, central_gravity=0.1, spring_length=150, spring_strength=0.0015, damping=0.09, overlap=0.8)


    template = net.templateEnv.get_template(net.path)
    net.template = template
    return net

In [104]:
results_env = results[(results.donor!="environment") & (results.receiver!="environment")]
results_env = results_env[["donor","receiver","frequency"]]
# Find mean of frequencies
results_grouped_rec_env = results_env.groupby(["donor","receiver"]).mean()

In [105]:
network_data = results_grouped_rec_env["frequency"].copy()

**Network topological properties**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Degree centrality**: The fraction of nodes that node i is connected to. 
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes.

In [106]:
centrality_metrics_SP = centrality_metrics(network_data).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Eubacterium_rectale_ATCC_33656,0.000000,0.043219
Blautia_hydrogenotrophica_DSM_10507,0.000000,0.045856
Collinsella_aerofaciens_ATCC_25986,0.000000,0.049145
Bacteroides_ovatus_ATCC_8483,0.000000,0.052995
Desulfovibrio_piger_ATCC_29098,0.000000,0.053885
Bacteroides_vulgatus_ATCC_8482,0.000000,0.058936
Bacteroides_uniformis_ATCC_8492,0.000000,0.063807
Bacteroides_thetaiotaomicron_VPI_5482,0.000000,0.065629
Prevotella_copri_CB7_DSM_18205,0.054545,0.071265
Eggerthella_lenta_DSM_2243,0.109091,0.076256


In [107]:
centrality_metrics(network_data,SP_centrality=False).sort_values("degree (data)")

,degree (data),eigenvector (data)
Blautia_hydrogenotrophica_DSM_10507,0.401954,0.152918
Eubacterium_rectale_ATCC_33656,0.452118,0.162315
Desulfovibrio_piger_ATCC_29098,0.474214,0.179885
Bacteroides_ovatus_ATCC_8483,0.499656,0.193882
Collinsella_aerofaciens_ATCC_25986,0.515544,0.195496
Bacteroides_vulgatus_ATCC_8482,0.655904,0.254008
Bacteroides_thetaiotaomicron_VPI_5482,0.752520,0.301610
Bacteroides_uniformis_ATCC_8492,0.775833,0.287483
Prevotella_copri_CB7_DSM_18205,0.825378,0.335594
Eggerthella_lenta_DSM_2243,0.870715,0.341528


In [108]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

In [109]:
net = frequency_network(network_data,closeness_centrality_met=closeness_centrality_met)
net.show("example.html")

example.html


In [111]:
net = frequency_network(network_data,closeness_centrality_met=closeness_centrality_met,cutoff=0.1)
net.show("example.html")

example.html


### Sum mass

In [220]:
def frequency_network(network_data,closeness_centrality_met,cutoff=0):
    net = Network(directed=True)

    nodes = list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique())))
    color = [colorFader(b1,b2,closeness_centrality_met[node]) for node in nodes]
    value = [(closeness_centrality_met[node]*10)**2 for node in nodes]
    net.add_nodes(nodes,color=color,value=value)
    
    
    maximum_corr = network_data.max()
    for index,value in network_data.items():
        if value>cutoff:
            net.add_edge(index[0],index[1],width=(value)*100,color=c2) #colorFader(c1,,value/maximum_corr)

    net.barnes_hut(gravity=-1000, central_gravity=0.1, spring_length=150, spring_strength=0.0015, damping=0.09, overlap=0.8)


    template = net.templateEnv.get_template(net.path)
    net.template = template
    return net

In [221]:
results_env = results[(results.donor!="environment") & (results.receiver!="environment")]
#results_env = results_env[results_env.frequency>0.1]

results_env = results_env[["donor","receiver","mass_rate*frequency"]]
# Find mean of frequencies
results_grouped_rec_env = results_env.groupby(["donor","receiver"]).sum()

In [222]:
network_data = results_grouped_rec_env["mass_rate*frequency"].copy()

**Network topological properties**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Degree centrality**: The fraction of nodes that node i is connected to. 
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes.

In [223]:
centrality_metrics_SP = centrality_metrics(network_data).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Desulfovibrio_piger_ATCC_29098,0.000000,0.000246
Prevotella_copri_CB7_DSM_18205,0.000000,0.000358
Eubacterium_rectale_ATCC_33656,0.000000,0.000685
Blautia_hydrogenotrophica_DSM_10507,0.000000,0.000888
Collinsella_aerofaciens_ATCC_25986,0.000000,0.001144
Eggerthella_lenta_DSM_2243,0.000000,0.001166
Bacteroides_vulgatus_ATCC_8482,0.000000,0.001360
Clostridium_hiranonis_TO_931_DSM_13275,0.327273,0.001362
Bacteroides_uniformis_ATCC_8492,0.072727,0.001389
Bacteroides_ovatus_ATCC_8483,0.109091,0.001392


In [224]:
centrality_metrics(network_data,SP_centrality=False).sort_values("degree (data)")

,degree (data),eigenvector (data)
Desulfovibrio_piger_ATCC_29098,0.000921,0.001405
Prevotella_copri_CB7_DSM_18205,0.001350,0.002597
Eubacterium_rectale_ATCC_33656,0.005992,0.009851
Blautia_hydrogenotrophica_DSM_10507,0.006414,0.012037
Eggerthella_lenta_DSM_2243,0.018321,0.016076
Collinsella_aerofaciens_ATCC_25986,0.019542,0.020223
Clostridium_hiranonis_TO_931_DSM_13275,0.042217,0.070634
Bacteroides_vulgatus_ATCC_8482,0.084324,0.190763
Faecalibacterium_prausnitzii_A2_165,0.183213,0.387476
Bacteroides_ovatus_ATCC_8483,0.208373,0.498934


In [225]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

In [226]:
net = frequency_network(network_data,closeness_centrality_met=closeness_centrality_met)
net.show("example.html")

example.html


In [229]:
net = frequency_network(network_data,closeness_centrality_met=closeness_centrality_met,cutoff=1e-3)
net.show("example.html")

example.html


In [127]:
results[(results.receiver=="Clostridium_hiranonis_TO_931_DSM_13275") & (results.donor!="environment") & (results.frequency>0.1)]

,donor,receiver,compound,mass_rate,rate,frequency,community,medium,mass_rate*frequency
121,Bacteroides_thetaiotaomicron_VPI_5482,Clostridium_hiranonis_TO_931_DSM_13275,M_cholate__91__e__93__,4.718934e-03,1.157845e-02,0.246,BV-,LB_extend,1.160858e-03
127,Bacteroides_thetaiotaomicron_VPI_5482,Clostridium_hiranonis_TO_931_DSM_13275,M_C02528__91__e__93__,4.627037e-03,1.181685e-02,0.214,BV-,LB_extend,9.901859e-04
201,Eggerthella_lenta_DSM_2243,Clostridium_hiranonis_TO_931_DSM_13275,M_cholate__91__e__93__,2.018753e-03,4.953242e-03,0.373,BV-,LB_extend,7.529948e-04
203,Collinsella_aerofaciens_ATCC_25986,Clostridium_hiranonis_TO_931_DSM_13275,M_cholate__91__e__93__,1.966675e-03,4.825464e-03,0.199,BV-,LB_extend,3.913684e-04
464,Bacteroides_uniformis_ATCC_8492,Clostridium_hiranonis_TO_931_DSM_13275,M_fru__91__e__93__,3.638388e-04,2.019583e-03,0.520,BV-,LB_extend,1.891962e-04
...,...,...,...,...,...,...,...,...,...
27624,Bacteroides_ovatus_ATCC_8483,Clostridium_hiranonis_TO_931_DSM_13275,M_so4__91__e__93__,4.958543e-07,5.161783e-06,0.471,CA-,LB_extend,2.335474e-07
27625,Bacteroides_thetaiotaomicron_VPI_5482,Clostridium_hiranonis_TO_931_DSM_13275,M_so4__91__e__93__,4.923801e-07,5.125617e-06,0.354,CA-,LB_extend,1.743026e-07
27846,Bacteroides_vulgatus_ATCC_8482,Clostridium_hiranonis_TO_931_DSM_13275,M_thymd__91__e__93__,6.983795e-08,2.883149e-07,1.000,CA-,LB_extend,6.983795e-08
27884,Prevotella_copri_CB7_DSM_18205,Clostridium_hiranonis_TO_931_DSM_13275,M_for__91__e__93__,2.432381e-08,5.403203e-07,0.116,CA-,LB_extend,2.821562e-09


In [128]:
network_data

donor                           receiver                              
Bacteroides_ovatus_ATCC_8483    Bacteroides_thetaiotaomicron_VPI_5482     0.571117
                                Bacteroides_uniformis_ATCC_8492           0.528146
                                Bacteroides_vulgatus_ATCC_8482            0.471814
                                Blautia_hydrogenotrophica_DSM_10507       0.528200
                                Clostridium_hiranonis_TO_931_DSM_13275    0.582841
                                                                            ...   
Prevotella_copri_CB7_DSM_18205  Clostridium_hiranonis_TO_931_DSM_13275    0.474739
                                Collinsella_aerofaciens_ATCC_25986        0.170750
                                Desulfovibrio_piger_ATCC_29098            0.231333
                                Eggerthella_lenta_DSM_2243                0.169300
                                Faecalibacterium_prausnitzii_A2_165       0.156758
Name: frequency,